# Fine-Tuning Qwen2-0.5B with Multiple PEFT Methods

**Strategy-conditioned counter-speech generation on IntentCONANv2**

This notebook implements and compares 4 fine-tuning methods:
1. **Full Fine-Tuning** - Update all model parameters
2. **LoRA** - Low-Rank Adaptation
3. **QLoRA** - Quantized LoRA (4-bit)
4. **Prefix-Tuning** - Trainable prefix embeddings

All methods use **HuggingFace Transformers + PEFT** for maximum reproducibility.

## Cell 1: Install Dependencies

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl
!pip install -q evaluate rouge-score nltk bert-score torchmetrics textstat detoxify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.4 MB/s eta 0:00:00


## Cell 2: Imports and Configuration

In [ ]:
import torch
import gc
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    PrefixTuningConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)
from datasets import load_dataset
from tqdm import tqdm
import numpy as np

# Configuration
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
MAX_SEQ_LENGTH = 2048

# Training configuration
QUICK_TEST_MODE = False  # Set to True for quick testing with max_steps
MAX_STEPS = 100  # Only used if QUICK_TEST_MODE=True
NUM_TRAIN_EPOCHS = 1  # Default: train on full dataset for 1 epoch

LEARNING_RATE = 5e-5
SEED = 42
TEST_SAMPLE_SIZE = None
GEN_BATCH_SIZE = 4

# LoRA/QLoRA config
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Set random seeds
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Cell 3: Load and Prepare Dataset

In [ ]:
import os
from huggingface_hub import login

# ========================================
# AUTHENTICATION (optional): read token from the environment
# Get your token from: https://huggingface.co/settings/tokens
# ========================================
HF_TOKEN = os.environ.get("HF_TOKEN")  # set in your environment / Colab secrets; never hardcode

# Login to HuggingFace
print("Authenticating with HuggingFace...")
if HF_TOKEN:
    login(token=HF_TOKEN)
print("Authentication successful!")

Authenticating with HuggingFace...
Authentication successful!


In [ ]:
# Load IntentCONANv2 dataset
dataset = load_dataset("Aswini123/IntentCONANv2")

print(f"Dataset: {dataset}")
print(f"Columns: {dataset['train'].column_names}")
print(f"Sample: {dataset['train'][0]}")

# Split 90% train, 10% test
full_data = dataset['train']
print(f"\nTotal samples: {len(full_data)}")

split_data = full_data.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split_data['train']
test_dataset = split_data['test']

# Subsample test set for faster evaluation
if TEST_SAMPLE_SIZE and len(test_dataset) > TEST_SAMPLE_SIZE:
    test_dataset = test_dataset.shuffle(seed=SEED).select(range(TEST_SAMPLE_SIZE))
    print(f"Subsampled test set to {TEST_SAMPLE_SIZE} samples for faster evaluation")

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Dataset: DatasetDict({
    train: Dataset({
        features: ['hatespeech', 'csType', 'counterspeech', 'hs_id', 'id', 'hatespeechTarget', 'speakerIntent', 'powerDynamics', 'hatespeechImplication', 'targetGroupCognitiveReaction', 'targetGroupEmotionalReaction', 'hatespeechOffensiveness'],
        num_rows: 9532
    })
    validation: Dataset({
        features: ['hatespeech', 'csType', 'counterspeech', 'hs_id', 'id', 'hatespeechTarget', 'speakerIntent', 'powerDynamics', 'hatespeechImplication', 'targetGroupCognitiveReaction', 'targetGroupEmotionalReaction', 'hatespeechOffensiveness'],
        num_rows: 1470
    })
    test: Dataset({
        features: ['hatespeech', 'csType', 'counterspeech', 'hs_id', 'id', 'hatespeechTarget', 'speakerIntent', 'powerDynamics', 'hatespeechImplication', 'targetGroupCognitiveReaction', 'targetGroupEmotionalReaction', 'hatespeechOffensiveness'],
        num_rows: 2971
    })
})
Columns: ['hatespeech', 'csType', 'counterspeech', 'hs_id', 'id', 'hatespeechTa

## Cell 4: Define Prompt Templates and Helper Functions

In [ ]:
# Prompt templates
PROMPT_TEMPLATE = """Below is a hate speech message and a required counter-speech strategy. Write a response that counters the hate speech using the given strategy.

### Hate Speech:
{hatespeech}

### Strategy:
{csType}

### Counter-speech Response:
{counterspeech}"""

INFERENCE_TEMPLATE = """Below is a hate speech message and a required counter-speech strategy. Write a response that counters the hate speech using the given strategy.

### Hate Speech:
{hatespeech}

### Strategy:
{csType}

### Counter-speech Response:
"""

def print_gpu_memory(stage):
    """Print current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"[{stage}] GPU Memory - Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")

def cleanup_memory():
    """Clean up GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print_gpu_memory("After cleanup")

def tokenize_data(examples, tokenizer):
    """Tokenize dataset for training."""
    texts = []
    EOS_TOKEN = tokenizer.eos_token

    for hs, cs_type, cs in zip(examples['hatespeech'], examples['csType'], examples['counterspeech']):
        text = PROMPT_TEMPLATE.format(
            hatespeech=hs,
            csType=cs_type,
            counterspeech=cs
        ) + EOS_TOKEN
        texts.append(text)

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False
    )
    tokenized["labels"] = [ids[:] for ids in tokenized["input_ids"]]
    return tokenized

# Prepare test data
test_prompts = [
    INFERENCE_TEMPLATE.format(hatespeech=ex['hatespeech'], csType=ex['csType'])
    for ex in test_dataset
]
ground_truth_responses = [ex['counterspeech'] for ex in test_dataset]
train_texts_for_novelty = [ex['counterspeech'] for ex in train_dataset]

# Storage for all generations
ALL_GENERATIONS = {}

print(f"Prepared {len(test_prompts)} test prompts")

Prepared 954 test prompts


## Cell 5: Evaluation Metrics Setup

In [ ]:
import evaluate
from bert_score import score as bert_score_fn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from textstat import flesch_reading_ease
from detoxify import Detoxify

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load metrics
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")
gleu_metric = evaluate.load("google_bleu")

# Load toxicity model
toxicity_model = Detoxify('original')

print("Evaluation metrics ready!")

def calculate_metrics(predictions, references, train_texts):
    """Calculate all required evaluation metrics."""
    results = {}

    # BLEU
    bleu = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    results['BLEU'] = bleu['bleu']

    # ROUGE
    rouge = rouge_metric.compute(predictions=predictions, references=references)
    results['ROUGE-1'] = rouge['rouge1']
    results['ROUGE-2'] = rouge['rouge2']
    results['ROUGE-L'] = rouge['rougeL']

    # METEOR
    meteor = meteor_metric.compute(predictions=predictions, references=references)
    results['METEOR'] = meteor['meteor']

    # GLEU
    gleu = gleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    results['GLEU'] = gleu['google_bleu']

    # Repetition Rate
    def repetition_rate(text):
        words = text.lower().split()
        if len(words) == 0:
            return 0
        return 1 - len(set(words)) / len(words)

    results['Repetition Rate'] = np.mean([repetition_rate(p) for p in predictions])

    # Flesch Reading Ease
    def safe_flesch(text):
        try:
            return flesch_reading_ease(text)
        except:
            return 0

    results['Flesch Reading Ease'] = np.mean([safe_flesch(p) for p in predictions])

    # CoSIM (TF-IDF cosine similarity)
    try:
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(predictions + references)
        pred_vectors = tfidf_matrix[:len(predictions)]
        ref_vectors = tfidf_matrix[len(predictions):]
        cosim_scores = [cosine_similarity(pred_vectors[i:i+1], ref_vectors[i:i+1])[0][0]
                       for i in range(len(predictions))]
        results['CoSIM'] = np.mean(cosim_scores)
    except:
        results['CoSIM'] = 0.0

    # BERT Score
    try:
        P, R, F1 = bert_score_fn(predictions, references, lang='en', verbose=False)
        results['BERT Score'] = F1.mean().item()
    except:
        results['BERT Score'] = 0.0

    # Toxicity (using Detoxify - lower is better)
    try:
        toxicity_scores = toxicity_model.predict(predictions)
        results['Toxicity'] = np.mean(toxicity_scores['toxicity'])
    except:
        results['Toxicity'] = 0.0

    # Novelty (1 - max similarity with training set)
    try:
        train_sample = train_texts[:500]  # Use subset for speed
        vectorizer = TfidfVectorizer()
        all_texts = predictions + train_sample
        tfidf_matrix = vectorizer.fit_transform(all_texts)
        pred_vectors = tfidf_matrix[:len(predictions)]
        train_vectors = tfidf_matrix[len(predictions):]

        novelty_scores = []
        for i in range(len(predictions)):
            sims = cosine_similarity(pred_vectors[i:i+1], train_vectors)[0]
            novelty_scores.append(1 - np.max(sims))
        results['Novelty'] = np.mean(novelty_scores)
    except:
        results['Novelty'] = 0.0

    # Diversity (avg pairwise distance)
    try:
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(predictions)
        similarity_matrix = cosine_similarity(tfidf_matrix)
        # Get upper triangle (excluding diagonal)
        n = len(predictions)
        upper_triangle = similarity_matrix[np.triu_indices(n, k=1)]
        results['Diversity'] = 1 - np.mean(upper_triangle)
    except:
        results['Diversity'] = 0.0

    return results

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:18<00:00, 24.1MB/s]


BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation metrics ready!


## Cell 6: Method 1 - Full Fine-Tuning

In [ ]:
print("\n" + "="*60)
print("METHOD 1: Full Fine-Tuning (All Parameters)")
print("="*60)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = 'left'  # Required for decoder-only models

# Check bf16 support
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float32
print(f"Using bf16: {use_bf16}, dtype: {compute_dtype}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Print trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} (100%)")

# Tokenize dataset
formatted_train = train_dataset.map(
    lambda x: tokenize_data(x, tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names
)

print_gpu_memory("After loading model")

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="outputs_full",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=NUM_TRAIN_EPOCHS if not QUICK_TEST_MODE else None,
    max_steps=MAX_STEPS if QUICK_TEST_MODE else -1,
    learning_rate=LEARNING_RATE,
    bf16=use_bf16,
    fp16=False,  # Disable fp16 to avoid gradient unscaling issues
    logging_steps=10,
    save_strategy="no",
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=10,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,  # Memory efficiency for full fine-tuning
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    data_collator=data_collator,
)

# Train
print("\nStarting training...")
trainer.train()
print_gpu_memory("After training")

# Generate responses
print("\nGenerating test responses...")
model.eval()
full_ft_responses = []

for i in tqdm(range(0, len(test_prompts), GEN_BATCH_SIZE), desc="Generating"):
    batch_prompts = test_prompts[i:i+GEN_BATCH_SIZE]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    for j, output in enumerate(outputs):
        full_response = tokenizer.decode(output, skip_special_tokens=True)
        if "### Counter-speech Response:" in full_response:
            response = full_response.split("### Counter-speech Response:")[-1].strip()
        else:
            response = full_response[len(batch_prompts[j]):].strip()
        full_ft_responses.append(response)

ALL_GENERATIONS["Full Fine-Tuning"] = full_ft_responses

print(f"\nSample generation:")
print(f"Input Prompt: {test_prompts[0]}")
print(f"Generated: {full_ft_responses[0]}")
print(f"Ground truth: {ground_truth_responses[0]}")

# Cleanup
del model, tokenizer, trainer, formatted_train
cleanup_memory()


METHOD 1: Full Fine-Tuning (All Parameters)


`torch_dtype` is deprecated! Use `dtype` instead!


Using bf16: True, dtype: torch.bfloat16


Total parameters: 494,032,768
Trainable parameters: 494,032,768 (100%)


[After loading model] GPU Memory - Allocated: 0.92 GB, Reserved: 0.93 GB

Starting training...


Step,Training Loss
10,2.096353
20,1.395410
30,1.336077
40,1.283130
50,1.327352
60,1.286288
70,1.168098
80,1.208536
90,1.201212
100,1.192950


[After training] GPU Memory - Allocated: 2.79 GB, Reserved: 5.32 GB

Generating test responses...


Generating: 100%|██████████| 239/239 [14:59<00:00,  3.76s/it]



Sample generation:
Generated: Muslims are not a problem in our country. They have contributed greatly to our society.
Ground truth: For all the islam is not a joke nonsense, you know it is a grave error to connect fundamentalists with islam and all the people who believe in it.
[After cleanup] GPU Memory - Allocated: 0.94 GB, Reserved: 0.96 GB


## Cell 7: Method 2 - LoRA

In [ ]:
print("\n" + "="*60)
print("METHOD 2: LoRA (Low-Rank Adaptation)")
print("="*60)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = 'left'  # Required for decoder-only models

# Check bf16 support
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float32
print(f"Using bf16: {use_bf16}, dtype: {compute_dtype}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize dataset
formatted_train = train_dataset.map(
    lambda x: tokenize_data(x, tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names
)

print_gpu_memory("After loading model")

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="outputs_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=NUM_TRAIN_EPOCHS if not QUICK_TEST_MODE else None,
    max_steps=MAX_STEPS if QUICK_TEST_MODE else -1,
    learning_rate=LEARNING_RATE,
    bf16=use_bf16,
    fp16=False,
    logging_steps=10,
    save_strategy="no",
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=10,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    data_collator=data_collator,
)

# Train
print("\nStarting training...")
trainer.train()
print_gpu_memory("After training")

# Generate responses
print("\nGenerating test responses...")
model.eval()
lora_responses = []

for i in tqdm(range(0, len(test_prompts), GEN_BATCH_SIZE), desc="Generating"):
    batch_prompts = test_prompts[i:i+GEN_BATCH_SIZE]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    for j, output in enumerate(outputs):
        full_response = tokenizer.decode(output, skip_special_tokens=True)
        if "### Counter-speech Response:" in full_response:
            response = full_response.split("### Counter-speech Response:")[-1].strip()
        else:
            response = full_response[len(batch_prompts[j]):].strip()
        lora_responses.append(response)

ALL_GENERATIONS["LoRA"] = lora_responses

print(f"\nSample generation:")
print(f"Input Prompt: {test_prompts[0]}")
print(f"Generated: {lora_responses[0]}")
print(f"Ground truth: {ground_truth_responses[0]}")

# Cleanup
del model, tokenizer, trainer, formatted_train
cleanup_memory()


METHOD 2: LoRA (Low-Rank Adaptation)
Using bf16: True, dtype: torch.bfloat16


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
[After loading model] GPU Memory - Allocated: 1.89 GB, Reserved: 1.90 GB

Starting training...


Step,Training Loss
10,3.059085
20,2.383227
30,1.898795
40,1.608492
50,1.550003
60,1.574587
70,1.454367
80,1.407044
90,1.379054
100,1.362581


[After training] GPU Memory - Allocated: 1.04 GB, Reserved: 3.04 GB

Generating test responses...


Generating: 100%|██████████| 239/239 [25:29<00:00,  6.40s/it]



Sample generation:
Generated: I understand your concerns about discrimination based on religion, but it's important to approach this topic with empathy and respect for all individuals. It's crucial to recognize that not everyone shares our values or beliefs. Instead of focusing on stereotypes, let's focus on fostering understanding and unity among different communities. By embracing diversity, we can build a more inclusive society where everyone feels valued and respected.
Ground truth: For all the islam is not a joke nonsense, you know it is a grave error to connect fundamentalists with islam and all the people who believe in it.
[After cleanup] GPU Memory - Allocated: 0.97 GB, Reserved: 1.90 GB


## Cell 8: Method 3 - QLoRA

In [ ]:
print("\n" + "="*60)
print("METHOD 3: QLoRA (Quantized LoRA - 4-bit)")
print("="*60)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = 'left'  # Required for decoder-only models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Check bf16 support
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"Using bf16: {use_bf16}, compute_dtype: {compute_dtype}")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0}
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize dataset
formatted_train = train_dataset.map(
    lambda x: tokenize_data(x, tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names
)

print_gpu_memory("After loading model")

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="outputs_qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=NUM_TRAIN_EPOCHS if not QUICK_TEST_MODE else None,
    max_steps=MAX_STEPS if QUICK_TEST_MODE else -1,
    learning_rate=LEARNING_RATE,
    bf16=use_bf16,
    fp16=False,
    logging_steps=10,
    save_strategy="no",
    optim="adamw_torch",  # Faster than paged_adamw_8bit for small models like 0.5B
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=10,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    data_collator=data_collator,
)

# Train
print("\nStarting training...")
trainer.train()
print_gpu_memory("After training")

# Generate responses
print("\nGenerating test responses...")
model.eval()
qlora_responses = []

for i in tqdm(range(0, len(test_prompts), GEN_BATCH_SIZE), desc="Generating"):
    batch_prompts = test_prompts[i:i+GEN_BATCH_SIZE]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    for j, output in enumerate(outputs):
        full_response = tokenizer.decode(output, skip_special_tokens=True)
        if "### Counter-speech Response:" in full_response:
            response = full_response.split("### Counter-speech Response:")[-1].strip()
        else:
            response = full_response[len(batch_prompts[j]):].strip()
        qlora_responses.append(response)

ALL_GENERATIONS["QLoRA"] = qlora_responses

print(f"\nSample generation:")
print(f"Input Prompt: {test_prompts[0]}")
print(f"Generated: {qlora_responses[0]}")
print(f"Ground truth: {ground_truth_responses[0]}")

# Cleanup
del model, tokenizer, trainer, formatted_train
cleanup_memory()


METHOD 3: QLoRA (Quantized LoRA - 4-bit)
Using bf16: True, compute_dtype: torch.bfloat16


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
[After loading model] GPU Memory - Allocated: 1.69 GB, Reserved: 2.46 GB

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,3.174401
20,2.359097
30,1.775295
40,1.606239
50,1.572240
60,1.532540
70,1.456486
80,1.397343
90,1.384689
100,1.373755


[After training] GPU Memory - Allocated: 0.80 GB, Reserved: 2.56 GB

Generating test responses...


Generating: 100%|██████████| 239/239 [40:52<00:00, 10.26s/it]



Sample generation:
Generated: I understand your concerns about discrimination based on religion, but it's important to approach this topic with empathy and respect for all individuals. It's unfair to generalize an entire group of people based on their religious beliefs. Islam has been a diverse faith throughout history, and it's crucial to recognize that diversity enriches our society. Instead of focusing on stereotypes, let's focus on fostering understanding and promoting inclusivity. By embracing open dialogue and learning from each other, we can build bridges and create a more harmonious world where everyone feels valued and respected.
Ground truth: For all the islam is not a joke nonsense, you know it is a grave error to connect fundamentalists with islam and all the people who believe in it.
[After cleanup] GPU Memory - Allocated: 0.73 GB, Reserved: 2.43 GB


## Cell 9: Method 4 - Prefix-Tuning

In [ ]:
print("\n" + "="*60)
print("METHOD 4: Prefix-Tuning")
print("="*60)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = 'left'  # Required for decoder-only models

# Check bf16 support
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float32
print(f"Using bf16: {use_bf16}, dtype: {compute_dtype}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Apply Prefix-Tuning
prefix_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20,
    prefix_projection=True,
)

model = get_peft_model(model, prefix_config)
model.print_trainable_parameters()

# Explicitly set to training mode
model.train()

# Tokenize dataset
formatted_train = train_dataset.map(
    lambda x: tokenize_data(x, tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names
)

print_gpu_memory("After loading model")

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="outputs_prefix",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=NUM_TRAIN_EPOCHS if not QUICK_TEST_MODE else None,
    max_steps=MAX_STEPS if QUICK_TEST_MODE else -1,
    learning_rate=LEARNING_RATE,
    bf16=use_bf16,
    fp16=False,
    logging_steps=10,
    save_strategy="no",
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=10,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    data_collator=data_collator,
)

# Train
print("\nStarting training...")
trainer.train()
print_gpu_memory("After training")

# Generate responses
print("\nGenerating test responses...")
model.eval()
prefix_responses = []

for i in tqdm(range(0, len(test_prompts), GEN_BATCH_SIZE), desc="Generating"):
    batch_prompts = test_prompts[i:i+GEN_BATCH_SIZE]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    for j, output in enumerate(outputs):
        full_response = tokenizer.decode(output, skip_special_tokens=True)
        if "### Counter-speech Response:" in full_response:
            response = full_response.split("### Counter-speech Response:")[-1].strip()
        else:
            response = full_response[len(batch_prompts[j]):].strip()
        prefix_responses.append(response)

ALL_GENERATIONS["Prefix-Tuning"] = prefix_responses

print(f"\nSample generation:")
print(f"Input Prompt: {test_prompts[0]}")
print(f"Generated: {prefix_responses[0]}")
print(f"Ground truth: {ground_truth_responses[0]}")

# Cleanup
del model, tokenizer, trainer, formatted_train
cleanup_memory()


METHOD 4: Prefix-Tuning
Using bf16: True, dtype: torch.bfloat16


trainable params: 811,648 || all params: 494,844,416 || trainable%: 0.1640
[After loading model] GPU Memory - Allocated: 1.66 GB, Reserved: 2.43 GB

Starting training...


Step,Training Loss
10,3.582080
20,3.465752
30,3.401106
40,3.300790
50,3.248736
60,3.161497
70,2.991178
80,2.975873
90,2.919503
100,2.931005


[After training] GPU Memory - Allocated: 0.95 GB, Reserved: 3.06 GB

Generating test responses...


Generating:   0%|          | 0/239 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Generating: 100%|██████████| 239/239 [08:55<00:00,  2.24s/it]



Sample generation:
Generated: Islam is not a religion but a political ideology. It's important to understand that Islam is a complex belief system with its own set of principles and values. While it can sometimes lead to discrimination against certain groups, it also promotes tolerance and respect for all people regardless of their beliefs or backgrounds. Therefore, it's essential to approach discussions about Islam with an open mind and understanding.
Ground truth: For all the islam is not a joke nonsense, you know it is a grave error to connect fundamentalists with islam and all the people who believe in it.
[After cleanup] GPU Memory - Allocated: 0.94 GB, Reserved: 1.88 GB


## Cell 10: Calculate Metrics for All Methods

In [ ]:
print("\n" + "="*60)
print("EVALUATING ALL METHODS")
print("="*60)

import pandas as pd

all_results = {}

for method_name, predictions in ALL_GENERATIONS.items():
    print(f"\nCalculating metrics for {method_name}...")
    results = calculate_metrics(predictions, ground_truth_responses, train_texts_for_novelty)
    all_results[method_name] = results

    print(f"Results for {method_name}:")
    for metric, value in results.items():
        print(f"  {metric}: {value:.4f}")

# Create comparison table
results_df = pd.DataFrame(all_results).T
print("\n" + "="*60)
print("FINAL RESULTS COMPARISON")
print("="*60)
print(results_df.to_string())

# Save to CSV
results_df.to_csv("fine_tuning_comparison_results.csv")
print("\nResults saved to: fine_tuning_comparison_results.csv")


EVALUATING ALL METHODS

Calculating metrics for Full Fine-Tuning...


RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Results for Full Fine-Tuning:
  BLEU: 0.0924
  ROUGE-1: 0.2696
  ROUGE-2: 0.0874
  ROUGE-L: 0.1937
  METEOR: 0.2535
  GLEU: 0.1204
  Repetition Rate: 0.0993
  Flesch Reading Ease: 34.7368
  CoSIM: 0.1823
  BERT Score: 0.8833
  Toxicity: 0.0386
  Novelty: 0.5985
  Diversity: 0.8775

Calculating metrics for LoRA...


RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Results for LoRA:
  BLEU: 0.0830
  ROUGE-1: 0.2536
  ROUGE-2: 0.0776
  ROUGE-L: 0.1781
  METEOR: 0.2387
  GLEU: 0.1100
  Repetition Rate: 0.0908
  Flesch Reading Ease: 35.8575
  CoSIM: 0.1634
  BERT Score: 0.8804
  Toxicity: 0.0263
  Novelty: 0.6298
  Diversity: 0.8664

Calculating metrics for QLoRA...


RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Results for QLoRA:
  BLEU: 0.0844
  ROUGE-1: 0.2489
  ROUGE-2: 0.0755
  ROUGE-L: 0.1777
  METEOR: 0.2318
  GLEU: 0.1091
  Repetition Rate: 0.0898
  Flesch Reading Ease: 37.1280
  CoSIM: 0.1599
  BERT Score: 0.8800
  Toxicity: 0.0265
  Novelty: 0.6331
  Diversity: 0.8727

Calculating metrics for Prefix-Tuning...


RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Results for Prefix-Tuning:
  BLEU: 0.0220
  ROUGE-1: 0.1672
  ROUGE-2: 0.0296
  ROUGE-L: 0.1219
  METEOR: 0.1318
  GLEU: 0.0478
  Repetition Rate: 0.0945
  Flesch Reading Ease: 49.4449
  CoSIM: 0.1041
  BERT Score: 0.8714
  Toxicity: 0.0606
  Novelty: 0.7029
  Diversity: 0.9166

FINAL RESULTS COMPARISON
                      BLEU   ROUGE-1   ROUGE-2   ROUGE-L    METEOR      GLEU  Repetition Rate  Flesch Reading Ease     CoSIM  BERT Score  Toxicity   Novelty  Diversity
Full Fine-Tuning  0.092439  0.269618  0.087443  0.193714  0.253464  0.120423         0.099258            34.736757  0.182299    0.883349  0.038605  0.598461   0.877493
LoRA              0.082988  0.253597  0.077623  0.178095  0.238682  0.109963         0.090807            35.857490  0.163396    0.880445  0.026284  0.629827   0.866370
QLoRA             0.084400  0.248921  0.075543  0.177727  0.231780  0.109061         0.089766            37.127970  0.159896    0.880032  0.026490  0.633142   0.872701
Prefix-Tuning     0.022

## Cell 11: Summary and Analysis

In [ ]:
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

print("\nMethods Evaluated:")
print("1. Full Fine-Tuning - All model parameters updated")
print("2. LoRA - Low-rank adaptation of specific layers")
print("3. QLoRA - 4-bit quantized LoRA for memory efficiency")
print("4. Prefix-Tuning - Learnable prefix tokens")

print(f"\nDataset: IntentCONANv2")
print(f"Model: {MODEL_NAME}")
if QUICK_TEST_MODE:
    print(f"Training mode: QUICK TEST (max_steps={MAX_STEPS})")
else:
    print(f"Training mode: FULL EPOCH (num_train_epochs={NUM_TRAIN_EPOCHS})")
print(f"Test samples: {len(test_prompts)}")

print("\nKey Findings:")
# Find best method for each metric
for metric in results_df.columns:
    best_method = results_df[metric].idxmax()
    best_score = results_df[metric].max()
    print(f"  Best {metric}: {best_method} ({best_score:.4f})")

print("\n" + "="*60)
print("Notebook execution complete!")
print("="*60)


SUMMARY

Methods Evaluated:
1. Full Fine-Tuning - All model parameters updated
2. LoRA - Low-rank adaptation of specific layers
3. QLoRA - 4-bit quantized LoRA for memory efficiency
4. Prefix-Tuning - Learnable prefix tokens

Dataset: IntentCONANv2
Model: Qwen/Qwen2-0.5B-Instruct
Training mode: FULL EPOCH (num_train_epochs=1)
Test samples: 954

Key Findings:
  Best BLEU: Full Fine-Tuning (0.0924)
  Best ROUGE-1: Full Fine-Tuning (0.2696)
  Best ROUGE-2: Full Fine-Tuning (0.0874)
  Best ROUGE-L: Full Fine-Tuning (0.1937)
  Best METEOR: Full Fine-Tuning (0.2535)
  Best GLEU: Full Fine-Tuning (0.1204)
  Best Repetition Rate: Full Fine-Tuning (0.0993)
  Best Flesch Reading Ease: Prefix-Tuning (49.4449)
  Best CoSIM: Full Fine-Tuning (0.1823)
  Best BERT Score: Full Fine-Tuning (0.8833)
  Best Toxicity: Prefix-Tuning (0.0606)
  Best Novelty: Prefix-Tuning (0.7029)
  Best Diversity: Prefix-Tuning (0.9166)

Notebook execution complete!
